In [0]:
!pip install -q websocket-client confluent-kafka python-dotenv

!apt-get install openjdk-8-jdk-headless -qq > /dev/null

!wget -q https://archive.apache.org/dist/spark/spark-3.5.1/spark-3.5.1-bin-hadoop3.tgz
!tar xf spark-3.5.1-bin-hadoop3.tgz

# 4. Cài đặt PySpark và Findspark
!pip install -q findspark pyspark

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.1-bin-hadoop3"

print("✅ Đã cài đặt xong Java, Spark và các thư viện cần thiết!")

## RUN in colab because Databricks block DNS

In [0]:
# Cấu hình Confluent Kafka
KAFKA_SERVER = "YOUR_BOOTSTRAP_SERVER"
KAFKA_API_KEY = "YOUR_API_KEY"
KAFKA_API_SECRET = "YOUR_API_SECRET"

In [0]:
import findspark
findspark.init()

from pyspark.sql import SparkSession

# Khởi tạo Spark với package Kafka
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Realtime-Crypto-Lakehouse") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1") \
    .getOrCreate()

print("✅ Đã khởi tạo Spark Session thành công với Kafka Connector!")

In [0]:
import json
import time
import logging
import websocket
import threading
from confluent_kafka import Producer

# Thiết lập log hiển thị chi tiết hơn
logging.basicConfig(level=logging.DEBUG, format='%(asctime)s | %(levelname)s | %(message)s')
logger = logging.getLogger(__name__)


class ColabMultiStreamProducer:
    def __init__(self, streams):
        streams_joined = "/".join(streams)
        self.ws_url = f"wss://stream.binance.us:9443/stream?streams={streams_joined}"
        self.msg_count = 0

        self.producer = Producer({
            'bootstrap.servers': KAFKA_SERVER,
            'security.protocol': 'SASL_SSL',
            'sasl.mechanisms': 'PLAIN',
            'sasl.username': KAFKA_API_KEY,
            'sasl.password': KAFKA_API_SECRET,
            'client.id': 'colab-multi-producer',
            'linger.ms': 100,
            'error_cb': self.error_callback,
            'debug': 'broker,security'
        })

    def error_callback(self, err):
        logger.error(f"LỖI KAFKA CẤP HỆ THỐNG: {err}")

    def delivery_report(self, err, msg):
        if err is not None:
            pass

    def on_message(self, ws, message):
        try:
            raw_data = json.loads(message)
            stream_name = raw_data['stream']
            data = raw_data['data']

            self.msg_count += 1

            if "kline" in stream_name:
                kline = data['k']
                topic = "binance-kline-1m"
                payload = {
                    "event_time": data['E'],
                    "symbol": data['s'],
                    "start_time": kline['t'],
                    "open": float(kline['o']),
                    "close": float(kline['c']),
                    "high": float(kline['h']),
                    "low": float(kline['l']),
                    "volume": float(kline['v']),
                    "is_closed": kline['x']
                }
                if payload['is_closed']:
                    logger.info(f"📈 [KLINE] Đóng nến {payload['symbol']} | Giá: {payload['close']}")

            elif "aggTrade" in stream_name:
                topic = "binance-trades"
                payload = {
                    "event_time": data['E'],
                    "type": "trade",
                    "symbol": data['s'],
                    "price": float(data['p']),
                    "qty": float(data['q']),
                    "is_buyer_maker": data['m']
                }

            else:
                return

            self.producer.produce(
                topic=topic,
                key=data.get('s', 'UNKNOWN'),
                value=json.dumps(payload).encode('utf-8'),
                callback=self.delivery_report
            )
            self.producer.poll(0)

        except Exception as e:
            logger.error(f"Lỗi xử lý tin nhắn: {e}")

    def on_error(self, ws, error):
        logger.error(f"Lỗi WebSocket: {error}")

    def on_close(self, ws, close_status_code, close_msg):
        logger.warning(f"Đứt kết nối. Tổng cộng đã đẩy {self.msg_count} tin nhắn. Đang dọn dẹp Kafka...")
        self.producer.flush()

    def start(self):
        logger.info(f"🚀 Bắt đầu thu thập ĐA LUỒNG. Tổng thời gian dự kiến: 60 phút...")

        def progress_tracker(ws_app):
            total_minutes = 120
            for current_min in range(1, total_minutes + 1):
                time.sleep(60)
                # Chỉ in tiến độ mỗi 5 phút hoặc khi kết thúc
                if current_min % 5 == 0 or current_min == total_minutes:
                    logger.info(f"⏱️ Tiến độ: Đã chạy {current_min}/{total_minutes} phút. Bắt được {self.msg_count} dòng dữ liệu.")

            logger.warning(f"⏰ Đã hoàn thành mục tiêu {total_minutes} phút! Tự động ngắt kết nối...")
            ws_app.close()

        while True:
            ws = websocket.WebSocketApp(
                self.ws_url,
                on_message=self.on_message,
                on_error=self.on_error,
                on_close=self.on_close
            )

            timer_thread = threading.Thread(target=progress_tracker, args=(ws,))
            timer_thread.start()

            ws.run_forever(ping_interval=60, ping_timeout=10)
            break

if __name__ == "__main__":
    my_streams = [
        "btcusdt@kline_1m",
        "ethusdt@kline_1m",
        "solusdt@kline_1m",
        "bnbusdt@kline_1m",
        "xrpusdt@kline_1m",
        "adausdt@kline_1m",
        "btcusdt@aggTrade",
        "ethusdt@aggTrade",
        "solusdt@aggTrade",
        "bnbusdt@aggTrade",
        "xrpusdt@aggTrade",
        "adausdt@aggTrade"
    ]

    app = ColabMultiStreamProducer(streams=my_streams)
    try:
        app.start()
    except KeyboardInterrupt:
        print("\n")
        logger.info(f"🛑 Người dùng bấm dừng sớm! Tổng cộng bắt được {app.msg_count} tin nhắn.")
        app.producer.flush()
        logger.info("✅ Đã tắt an toàn!")